In [5]:
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# =========================
# CREATE PROJECT FOLDERS
# =========================
base_path = "AI_Business_Analytics_Project"
folders = ["data_raw", "data_cleaned", "sql_db", "python_analysis", "dashboards", "agents", "voice_ai"]
for folder in folders:
    os.makedirs(os.path.join(base_path, folder), exist_ok=True)
print("✅ Folders ready.")

# =========================
# SUPPLIERS TABLE (50 rows)
# =========================
supplier_names = [f"Supplier_{i}" for i in range(1, 51)]
supplier_cities = ["Mumbai", "Delhi", "Indore", "Bhopal", "Pune", "Ahmedabad"]
contact_types = ["Wholesale", "Distributor", "Manufacturer"]

suppliers = []
for i in range(1, 51):
    suppliers.append([
        f"S{i:03d}",
        supplier_names[i-1],
        random.choice(supplier_cities),
        random.choice(contact_types)
    ])

suppliers_df = pd.DataFrame(suppliers, columns=["supplier_id", "supplier_name", "city", "contact_type"])
suppliers_df.to_csv(os.path.join(base_path, "data_raw/suppliers.csv"), index=False)
print("✅ suppliers.csv created")

# =========================
# PRODUCTS TABLE (1200 rows + dirty values)
# =========================
categories = ["Groceries", "groceries", "Beverages", "Snacks", "Household", "Personal Care", "Stationery", "Dairy", "Frozen", "Bakery"]
brands = ["Nestle", "Patanjali", "HUL", "Dabur", "Amul", "Britannia", None, "amul", "Nestlé"]

products = []
for i in range(1, 1201):
    products.append([
        f"P{i:04d}",
        f"Product_{i}",
        random.choice(categories),
        random.choice(brands),
        f"S{random.randint(1,50):03d}",
        random.randint(20, 2500)
    ])

products_df = pd.DataFrame(products, columns=["product_id", "product_name", "category", "brand", "supplier_id", "unit_price"])
products_df = pd.concat([products_df, products_df.sample(20)], ignore_index=True)  # duplicates
products_df.to_csv(os.path.join(base_path, "data_raw/products.csv"), index=False)
print("✅ products.csv created")

# =========================
# INVENTORY TABLE (linked to products)
# =========================
inv_records = []
for idx, row in products_df.drop_duplicates('product_id').iterrows():
    restock = pd.to_datetime("2025-01-01") + pd.to_timedelta(random.randint(1, 250), unit='D')
    if random.random() < 0.05:
        restock = None

    stock = random.randint(-10, 800) if random.random() < 0.03 else random.randint(10, 800)
    reorder = None if random.random() < 0.04 else random.randint(20, 120)

    inv_records.append([
        f"I{idx+1:04d}",
        row['product_id'],
        stock,
        reorder,
        random.choice(["Mumbai", "Delhi", "Indore", "Bhopal", "Pune"]),
        restock
    ])

inventory_df = pd.DataFrame(inv_records, columns=["inventory_id", "product_id", "stock_qty", "reorder_level", "warehouse_location", "last_restock_date"])
inventory_df.to_csv(os.path.join(base_path, "data_raw/inventory.csv"), index=False)
print("✅ inventory.csv created")

# =========================
# SALES TABLE (60000 rows + dirty data)
# =========================
payment_modes = ["UPI", "upi", "Cash", "Card", "Net Banking", None]
regions = ["North", "South", "East", "West", "Central"]
start_date = datetime(2025,1,1)

sales_records = []
unique_products = products_df.drop_duplicates('product_id')

for i in range(1, 60001):
    prod = unique_products.sample(1).iloc[0]
    qty = random.randint(1, 10)
    if random.random() < 0.01:
        qty = random.randint(30, 100)

    order_date = start_date + timedelta(days=random.randint(0,364))
    order_date = order_date.strftime("%Y-%m-%d") if random.random() > 0.08 else order_date.strftime("%d/%m/%Y")

    sales_amount = qty * prod['unit_price']
    profit = round(sales_amount * random.uniform(-0.05, 0.25), 2)

    sales_records.append([
        i,
        order_date,
        prod['product_id'],
        f"C{random.randint(1000,9999)}",
        qty,
        random.choice(payment_modes),
        random.choice(regions),
        sales_amount,
        profit
    ])

sales_df = pd.DataFrame(sales_records, columns=["order_id", "order_date", "product_id", "customer_id", "quantity_sold", "payment_mode", "region", "sales_amount", "profit"])
sales_df = pd.concat([sales_df, sales_df.sample(100)], ignore_index=True)
sales_df.to_csv(os.path.join(base_path, "data_raw/sales.csv"), index=False)
print("✅ sales.csv created")

# =========================
# HOUSEHOLD EXPENSE TABLE (15000 rows + dirty data)
# =========================
exp_categories = ["Groceries", "Grocceries", "Electricity", "Rent", "Transport", "Medical", "Shopping", "Entertainment", "Education", "Fuel", "Internet"]
expense_payments = ["UPI", "upi", "Cash", "Credit Card", "Debit Card", None]
cities = ["Indore", "Bhopal", "Delhi", "Mumbai", "Pune", None]

expense_records = []
for i in range(1, 15001):
    dt = start_date + timedelta(days=random.randint(0,364))
    dt = dt.strftime("%Y-%m-%d") if random.random() > 0.07 else dt.strftime("%d-%m-%Y")
    cat = random.choice(exp_categories)

    amt = random.randint(100, 5000)
    if cat == "Rent":
        amt = random.randint(8000, 25000)
    if random.random() < 0.01:
        amt = random.randint(30000, 70000)

    expense_records.append([
        i,
        f"HH{random.randint(100,999)}",
        dt,
        cat,
        amt,
        random.choice(expense_payments),
        random.choice(cities)
    ])

expense_df = pd.DataFrame(expense_records, columns=["transaction_id", "household_id", "date", "category", "amount", "payment_mode", "city"])
expense_df = pd.concat([expense_df, expense_df.sample(50)], ignore_index=True)
expense_df.to_csv(os.path.join(base_path, "data_raw/household_expenses.csv"), index=False)
print("✅ household_expenses.csv created")

print("\n🎉 PROFESSIONAL SQL-READY DIRTY DATA WAREHOUSE GENERATED SUCCESSFULLY")

✅ Folders ready.
✅ suppliers.csv created
✅ products.csv created
✅ inventory.csv created
✅ sales.csv created
✅ household_expenses.csv created

🎉 PROFESSIONAL SQL-READY DIRTY DATA WAREHOUSE GENERATED SUCCESSFULLY
